# Task 2 - Unsupervised Learning: Association Rule Mining

This notebook applies association rule mining to discover traffic patterns that predict congestion levels.

**Approach:**
- Discretize continuous variables (time, traffic volume, weather)
- Apply Apriori algorithm with min_support=0.02 and min_confidence=0.6
- Extract rules predicting congestion levels
- Report top rules by lift value

## 1. Import Libraries and Load Script

In [19]:
import pandas as pd
import numpy as np
from mlxtend.frequent_patterns import apriori, association_rules
from mlxtend.preprocessing import TransactionEncoder
import matplotlib.pyplot as plt
import seaborn as sns

print("OK - Libraries imported")

OK - Libraries imported


In [20]:
def load_and_discretize(filepath):
    """Load preprocessed data and discretize for association rule mining."""
    df = pd.read_csv(filepath)
    df['date_time'] = pd.to_datetime(df['date_time'])
    df['hour'] = df['date_time'].dt.hour
    df['day_of_week'] = df['date_time'].dt.dayofweek
    
    # Discretize time period
    def time_period(h):
        if 6 <= h < 10:
            return 'morning_rush'
        elif 10 <= h < 16:
            return 'midday'
        elif 16 <= h < 20:
            return 'evening_rush'
        else:
            return 'night'
    
    df['time_period'] = df['hour'].apply(time_period)
    df['weekday_type'] = df['day_of_week'].apply(lambda x: 'weekday' if x < 5 else 'weekend')
    
    # Use existing weather_main (already in preprocessed data)
    df['weather_category'] = df['weather_main'].str.lower()
    
    # Use existing congestion_category (already created in preprocessing)
    df['congestion_category'] = df['congestion_category'].str.lower()
    
    return df

print("OK - Data loading function defined")

OK - Data loading function defined


In [21]:
def create_transactions(df):
    """Create transaction dataset for Apriori algorithm."""
    transactions = []
    for _, row in df.iterrows():
        transactions.append([
            f"time={row['time_period']}",
            f"weekday={row['weekday_type']}",
            f"weather={row['weather_category']}",
            f"congestion={row['congestion_category']}"
        ])
    return transactions

def run_mining(transactions, min_support=0.02, min_confidence=0.6):
    """Run Apriori algorithm and extract association rules."""
    te = TransactionEncoder()
    df_encoded = pd.DataFrame(te.fit(transactions).transform(transactions), columns=te.columns_)
    
    itemsets = apriori(df_encoded, min_support=min_support, use_colnames=True)
    if len(itemsets) == 0:
        return pd.DataFrame()
    
    rules = association_rules(itemsets, metric="confidence", min_threshold=min_confidence)
    if len(rules) == 0:
        return pd.DataFrame()
    
    rules = rules[rules['consequents'].apply(lambda x: any('congestion=' in str(i) for i in x))].copy()
    rules['antecedent_str'] = rules['antecedents'].apply(lambda x: ', '.join(list(x)))
    rules['consequent_str'] = rules['consequents'].apply(lambda x: ', '.join(list(x)))
    
    return rules.sort_values('lift', ascending=False).reset_index(drop=True)

print("OK - Mining functions defined")

OK - Mining functions defined


In [22]:
df = load_and_discretize('Metro_Interstate_Traffic_Volume_part3_preprocessed.csv')
transactions = create_transactions(df)
rules = run_mining(transactions, min_support=0.02, min_confidence=0.6)

print(f"Association rule mining complete:")
print(f"  Transactions: {len(df):,}")
print(f"  Rules found: {len(rules)}")

if len(rules) > 0:
    print(f"  Lift: [{rules['lift'].min():.2f}, {rules['lift'].max():.2f}]")
    top_rules = rules.head(15)

Association rule mining complete:
  Transactions: 48,187
  Rules found: 9
  Lift: [2.56, 3.42]


In [24]:
if len(rules) > 0:
    print(f"\n{'='*80}")
    print(f"TOP {min(15, len(rules))} ASSOCIATION RULES (by Lift)")
    print(f"{'='*80}")
    
    for idx, (_, row) in enumerate(top_rules.iterrows(), 1):
        print(f"\nRule {idx}:")
        print(f"  Antecedent: {row['antecedent_str']}")
        print(f"  Consequent: {row['consequent_str']}")
        print(f"  Support: {row['support']:.4f} | Confidence: {row['confidence']:.1%} | Lift: {row['lift']:.2f}x")
else:
    print("No rules found.")


TOP 9 ASSOCIATION RULES (by Lift)

Rule 1:
  Antecedent: weekday=weekday, time=morning_rush, weather=clear
  Consequent: congestion=severe
  Support: 0.0247 | Confidence: 85.4% | Lift: 3.42x

Rule 2:
  Antecedent: weather=clouds, time=midday, weekday=weekend
  Consequent: congestion=high
  Support: 0.0223 | Confidence: 82.7% | Lift: 3.31x

Rule 3:
  Antecedent: weekday=weekday, time=morning_rush
  Consequent: congestion=severe
  Support: 0.1003 | Confidence: 82.2% | Lift: 3.29x

Rule 4:
  Antecedent: weekday=weekday, weather=clouds, time=morning_rush
  Consequent: congestion=severe
  Support: 0.0270 | Confidence: 81.7% | Lift: 3.27x

Rule 5:
  Antecedent: time=midday, weekday=weekend
  Consequent: congestion=high
  Support: 0.0557 | Confidence: 80.4% | Lift: 3.22x

Rule 6:
  Antecedent: time=evening_rush, weekday=weekend
  Consequent: congestion=high
  Support: 0.0338 | Confidence: 73.3% | Lift: 2.93x

Rule 7:
  Antecedent: weather=mist, time=night
  Consequent: congestion=low
  Suppo

## Summary

**Association Rule Mining Complete**

### Key Findings:

1. **Rule Discovery**: Identified patterns in traffic data that predict congestion levels

2. **Top Rules**: The highest-lift rules show the strongest associations between conditions and congestion

3. **Pattern Insights**: Certain time periods combined with weather conditions predict congestion

4. **Practical Applications**: Use rules for congestion prediction and traffic management


### Thresholds Used:
- **Min Support**: 0.02
- **Min Confidence**: 0.6
- **Lift Metric**: Consequent likelihood increase given antecedent